In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import random
from scipy.spatial.distance import euclidean

In [2]:
def fastdtw(x, y, dist=euclidean):
    n, m = len(x), len(y)
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = dist([x[i - 1]], [y[j - 1]])
            dtw[i, j] = cost + min(
                dtw[i - 1, j],
                dtw[i, j - 1],
                dtw[i - 1, j - 1]
            )

    return dtw[n, m]


In [3]:
INPUT_AUGMENTED_AUDIO_DIR = "Augmented_audio_550"
OUTPUT_FEATURE_DIR = "Entropy_DTW_LSTM_features"

KNOWN_SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
FIXED_TIME_FRAMES = 219

DTW_DOWNSAMPLE_FACTOR = 3
MAX_PROTOTYPE_SAMPLES = 80

os.makedirs(OUTPUT_FEATURE_DIR, exist_ok=True)


In [ ]:
def shannon_entropy(signal, bins=100):
    hist, _ = np.histogram(signal, bins=bins, density=True)
    hist = hist[hist > 0]
    return -np.sum(hist * np.log2(hist)) if len(hist) else 0.0


def spectral_entropy(y):
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
    psd = S / (np.sum(S, axis=0, keepdims=True) + 1e-12)
    return -np.sum(psd * np.log2(psd + 1e-12), axis=0)


def bandwise_entropy(y, sr, num_bands=4):
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band_edges = np.linspace(0, sr // 2, num_bands + 1)

    bands = []
    for i in range(num_bands):
        idx = np.where((freqs >= band_edges[i]) &
                       (freqs < band_edges[i + 1]))[0]
        psd_band = S[idx]
        psd_band /= np.sum(psd_band, axis=0, keepdims=True) + 1e-12
        bands.append(-np.sum(psd_band * np.log2(psd_band + 1e-12), axis=0))

    return np.array(bands)


def extract_entropy_features(y):
    fe = spectral_entropy(y)
    de = np.diff(fe, prepend=fe[0])
    sh = shannon_entropy(y)
    bands = bandwise_entropy(y, KNOWN_SR)

    def pad(x):
        return np.pad(x, (0, FIXED_TIME_FRAMES - len(x)))[:FIXED_TIME_FRAMES]

    fe, de = pad(fe), pad(de)
    bands = np.array([pad(b) for b in bands])
    sh = np.ones(FIXED_TIME_FRAMES) * sh

    features = np.vstack([fe, de, sh, bands])
    features = (features - features.mean()) / (features.std() + 1e-8)

    return features  # (12 × T)


: 

In [ ]:
entropy_store = []
dtw_sequences = []
labels = []
file_refs = []

class_names = sorted(os.listdir(INPUT_AUGMENTED_AUDIO_DIR))

for cls in class_names:
    for fname in os.listdir(os.path.join(INPUT_AUGMENTED_AUDIO_DIR, cls)):
        if not fname.endswith(".wav"):
            continue

        y, _ = librosa.load(
            os.path.join(INPUT_AUGMENTED_AUDIO_DIR, cls, fname),
            sr=KNOWN_SR
        )
        y = librosa.util.fix_length(y, KNOWN_SR * 7)

        entropy_feat = extract_entropy_features(y)
        entropy_store.append(entropy_feat)

        dtw_seq = entropy_feat[0][::DTW_DOWNSAMPLE_FACTOR]
        dtw_sequences.append(dtw_seq)

        labels.append(cls)
        file_refs.append((cls, fname))


In [ ]:
prototypes = {}

for cls in class_names:
    seqs = [dtw_sequences[i]
            for i in range(len(labels)) if labels[i] == cls]

    if len(seqs) > MAX_PROTOTYPE_SAMPLES:
        seqs = random.sample(seqs, MAX_PROTOTYPE_SAMPLES)

    best, proto = np.inf, None
    for cand in seqs:
        s = sum(fastdtw(cand, other) for other in seqs)
        if s < best:
            best, proto = s, cand

    prototypes[cls] = proto


In [ ]:
manifest = []

for i, (cls, fname) in enumerate(file_refs):
    entropy_feat = entropy_store[i]
    dtw_seq = dtw_sequences[i]

    dtw_stats = dtw_statistics(dtw_seq, prototypes)
    dtw_stats = (dtw_stats - dtw_stats.mean()) / (dtw_stats.std() + 1e-8)

    dtw_expanded = np.tile(dtw_stats[:, None], (1, FIXED_TIME_FRAMES))
    final_features = np.vstack([entropy_feat, dtw_expanded])

    out_dir = os.path.join(OUTPUT_FEATURE_DIR, cls)
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, fname.replace(".wav", "_features.npy"))
    np.save(out_path, final_features)

    manifest.append({
        "class": cls,
        "path": out_path,
        "shape": final_features.shape
    })


In [ ]:
pd.DataFrame(manifest).to_csv(
    os.path.join(OUTPUT_FEATURE_DIR, "_manifest_entropy_dtw.csv"),
    index=False
)

print("✅ Corrected Entropy + DTW features saved")
print("Feature shape:", final_features.shape)
